In [ ]:
from minigrid.wrappers import ImgObsWrapper
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv

from feature_extractor import MinigridFeaturesExtractor
from minigrider import MiniGrider

from versioned_curriculum_trainer import VersionedCurriculumTrainer
from model_version_manager import ModelVersionManager

SEED = 1123
DEVICE = "cpu"  # PPO is meant to run on CPU
N_ENVS = 6
MODELS_BASE_DIR = "local_models"

In [ ]:
def create_ppo_cnn_model(env, device=DEVICE, verbose=1):
    policy_kwargs = {
        "features_extractor_class": MinigridFeaturesExtractor,
        "features_extractor_kwargs": {"features_dim": 128},
    }

    model = PPO(
        policy="CnnPolicy",
        env=env,
        policy_kwargs=policy_kwargs,
        verbose=verbose,
        device=device
    )

    return model

In [ ]:
def make_minigrid_vecenv(level_id, n_envs=N_ENVS, seed=SEED):
    env = make_vec_env(
        MiniGrider,
        n_envs=n_envs,
        vec_env_cls=SubprocVecEnv,
        wrapper_class=ImgObsWrapper,
        seed=seed,
        env_kwargs={"level_id": level_id}
    )
    
    return env

In [ ]:
version_manager = ModelVersionManager(base_dir=MODELS_BASE_DIR)

trainer = VersionedCurriculumTrainer(
    env_fn=make_minigrid_vecenv,
    model_fn=create_ppo_cnn_model,
    version_manager=version_manager,
    env_class=MiniGrider,
)

In [ ]:
trainer.train_all_levels(
    timesteps_per_level=10_000
)

In [ ]:
import numpy as np
from stable_baselines3 import PPO

def evaluate_model(
    model_path,
    env_fn,
    level_id,
    n_episodes=20,
    device=DEVICE,
):
    env = env_fn(level_id)

    model = PPO.load(model_path, env=env, device=device)

    rewards = []
    lengths = []
    successes = []

    for _ in range(n_episodes):
        obs = env.reset()
        done = False
        total_reward = 0
        steps = 0
        success = False

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, info = env.step(action)

            reward = reward[0]
            done = done[0]

            total_reward += reward
            steps += 1

            if done and reward > 0:
                success = True

        rewards.append(total_reward)
        lengths.append(steps)
        successes.append(success)

    env.close()

    return {
        "mean_reward": float(np.mean(rewards)),
        "std_reward": float(np.std(rewards)),
        "mean_length": float(np.mean(lengths)),
        "success_rate": float(np.mean(successes)),
    }


In [ ]:
model_level_id = 1
test_level_id = 1

model_path = f"{MODELS_BASE_DIR}/level_{model_level_id}/run_001.zip"

stats = evaluate_model(
    model_path=model_path,
    env_fn=make_minigrid_vecenv,
    level_id=test_level_id,
    n_episodes=50
)

print(stats)

In [ ]:
def visual_test_model(
    model_path,
    level_id,
    device=DEVICE
):
    env = MiniGrider(level_id, render_mode="human")
    env = ImgObsWrapper(env)

    model = PPO.load(model_path, env=env, device=device)

    obs, info = env.reset()
    while True:
        action, _ = model.predict(obs)
        obs, rewards, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            break

    env.close()

In [ ]:
model_level_id = 1
test_level_id = 1

model_path = f"{MODELS_BASE_DIR}/level_{model_level_id}/run_001.zip"

visual_test_model(
    model_path=model_path,
    level_id=test_level_id
)